## Daftar Isi

- [Impor Data dan Persiapan](#Impor-Data-dan-Persiapan)
- [Pembersihan dan Prapemrosesan Data](#Pembersihan-dan-Prapemrosesan-Data)
- [Analisis Data Eksploratif](#Analisis-Data-Eksploratif)
- [Pemodelan dan Prediksi](#Pemodelan-dan-Prediksi)
- [Kesimpulan dan Pekerjaan Mendatang](#Kesimpulan-dan-Pekerjaan-Mendatang)

In [ ]:
# Impor pustaka yang diperlukan dan nonaktifkan peringatan
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Fungsi bantu untuk menemukan lokasi file data
DATA_ROOTS = [
    Path('/kaggle/input'),
    Path('/kaggle/input'),
    Path('/kaggle/input/datasets')
]

def resolve_data_file(file_name=None, suffixes=None):
    suffixes = tuple(suffixes or [])
    for root in DATA_ROOTS:
        if not root.exists():
            continue
        matches = [p for p in root.rglob('*') if p.is_file()]
        if file_name:
            exact = [p for p in matches if p.name == file_name]
            if exact:
                return exact[0]
        if suffixes:
            typed = [p for p in matches if p.suffix.lower() in suffixes]
            if typed:
                return typed[0]
    raise FileNotFoundError(f'Tidak dapat menemukan file data: {file_name or suffixes}')

# Temukan lokasi file dataset menggunakan nama file yang diberikan
file_path = resolve_data_file(file_name='ai_student_impact_dataset (1).csv', suffixes=['.csv'])

# Muat dataset
df = pd.read_csv(file_path, encoding='ascii', sep=',')

# Gambaran singkat
print('Ukuran dataset:', df.shape)
df.head()

## Pembersihan dan Prapemrosesan Data

Dataset ini berisi berbagai fitur, mulai dari nilai numerik hingga informasi kategoris tentang penggunaan AI. Saya memeriksa nilai yang hilang, tipe data, dan ketidakkonsistenan yang dapat memengaruhi analisis. Pada tahap ini, saya juga mengonversi kolom boolean dan menetapkan tipe data yang sesuai jika memungkinkan.

In [ ]:
# Periksa nilai yang hilang
missing_vals = df.isnull().sum()
print('Nilai yang hilang pada setiap kolom:')
print(missing_vals)

# Ubah Paid_Subscription menjadi boolean jika belum
if df['Paid_Subscription'].dtype != 'bool':
    df['Paid_Subscription'] = df['Paid_Subscription'].astype(bool)

# Pastikan Year_of_Study bertipe kategoris
df['Year_of_Study'] = df['Year_of_Study'].astype('category')

# Tampilkan tipe data untuk pemeriksaan singkat
print('\nTipe data:')
print(df.dtypes)

## Analisis Data Eksploratif

Di sini, saya mengeksplorasi dataset secara visual menggunakan berbagai jenis grafik. Saya mulai dengan berfokus pada hubungan numerik, lalu menelaah distribusi kategoris. Tujuannya adalah mengidentifikasi korelasi menarik yang dapat membantu proses pemodelan. Perlu diperhatikan bahwa metode numerik hanya diterapkan pada kolom numerik.

In [ ]:
# Pilih hanya kolom numerik untuk analisis korelasi
numeric_df = df.select_dtypes(include=[np.number])
print('Kolom numerik:', numeric_df.columns.tolist())

# Buat peta panas korelasi jika terdapat empat kolom numerik atau lebih
if numeric_df.shape[1] >= 4:
    plt.figure(figsize=(10, 8))
    corr = numeric_df.corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Peta Panas Korelasi - Fitur Numerik')
    plt.show()
else:
    print('Jumlah kolom numerik tidak cukup untuk membuat peta panas korelasi yang bermakna.')

In [ ]:
# Pair plot untuk memvisualisasikan hubungan berpasangan
sns.pairplot(numeric_df)
plt.suptitle('Pair Plot Fitur Numerik', y=1.02)
plt.show()

# Histogram untuk variabel numerik utama
plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
sns.histplot(df['Pre_Semester_GPA'], kde=True, color='skyblue')
plt.title('Distribusi IPK Sebelum Semester')

plt.subplot(1, 3, 2)
sns.histplot(df['Post_Semester_GPA'], kde=True, color='salmon')
plt.title('Distribusi IPK Setelah Semester')

plt.subplot(1, 3, 3)
sns.histplot(df['Skill_Retention_Score'], kde=True, color='limegreen')
plt.title('Distribusi Skor Retensi Keterampilan')

plt.tight_layout()
plt.show()

# Grafik batang untuk kolom kategoris, misalnya kebijakan institusi
plt.figure(figsize=(10, 4))
sns.countplot(data=df, x='Institutional_Policy', palette='viridis')
plt.title('Jumlah untuk Berbagai Kebijakan Institusi')
plt.xticks(rotation=45)
plt.show()

# Box plot untuk memeriksa perbedaan distribusi IPK berdasarkan kategori jurusan
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='Major_Category', y='Post_Semester_GPA', palette='Set2')
plt.title('IPK Setelah Semester berdasarkan Kategori Jurusan')
plt.xticks(rotation=45)
plt.show()

## Pemodelan dan Prediksi

Saya kini membangun model prediksi untuk memperkirakan IPK akhir semester (*Post-Semester GPA*) berdasarkan fitur yang mencakup IPK sebelum semester (*Pre-Semester GPA*), jam penggunaan AI generatif per minggu, jam belajar konvensional, dan tingkat kecemasan saat ujian. Memprediksi hasil akademik bukanlah perkara sederhana (dan mungkin tidak seakurat bola ajaib peramal), tetapi model saya memberikan perkiraan kuantitatif. Kinerja model dievaluasi menggunakan skor R² dan RMSE. Humor kering boleh saja, tetapi statistik tetap tidak bisa ditawar.

In [ ]:
# Tentukan fitur dan variabel target
features = ['Pre_Semester_GPA', 'Weekly_GenAI_Hours', 'Traditional_Study_Hours', 'Anxiety_Level_During_Exams']
target = 'Post_Semester_GPA'

# Periksa apakah terdapat nilai yang hilang pada fitur terpilih
print('Nilai yang hilang pada fitur terpilih:')
print(df[features].isnull().sum())

# Bagi dataset menjadi data latih dan data uji
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Buat dan latih model regresi
model = LinearRegression()
model.fit(X_train, y_train)

# Lakukan prediksi pada data uji
y_pred = model.predict(X_test)

# Evaluasi kinerja model
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f'RMSE: {rmse:.2f}')
print(f'Skor R²: {r2:.2f}')

# Buat grafik perbandingan prediksi dengan nilai sebenarnya
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.7, color='teal')
plt.xlabel('IPK Setelah Semester Sebenarnya')
plt.ylabel('IPK Setelah Semester yang Diprediksi')
plt.title('Predicted vs IPK Setelah Semester Sebenarnya')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)  
plt.show()

## Perbandingan Model: Decision Tree dan Random Forest

Selain regresi linear, saya menambahkan **Decision Tree Regressor** dan **Random Forest Regressor** untuk membandingkan performa beberapa model dalam memprediksi IPK setelah semester. Ketiga model menggunakan fitur, target, dan pembagian data latih-uji yang sama agar perbandingannya lebih adil.

Evaluasi menggunakan **RMSE** (semakin rendah semakin baik) dan **R²** (semakin tinggi semakin baik). Hasil aktual akan muncul setelah sel kode dijalankan.

In [ ]:
# Impor model dan metrik evaluasi yang diperlukan
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Tentukan fitur dan variabel target yang sama untuk semua model
features = [
    'Pre_Semester_GPA',
    'Weekly_GenAI_Hours',
    'Traditional_Study_Hours',
    'Anxiety_Level_During_Exams'
]
target = 'Post_Semester_GPA'

# Siapkan data dan hapus baris yang memiliki nilai kosong pada kolom terpilih
model_data = df[features + [target]].dropna()
X = model_data[features]
y = model_data[target]

# Bagi data dengan pembagian yang sama agar hasil model dapat dibandingkan
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Siapkan model yang akan dibandingkan
models = {
    'Regresi Linear': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}

# Latih setiap model dan hitung metrik evaluasinya
results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_model = model.predict(X_test)
    predictions[name] = y_pred_model
    results.append({
        'Model': name,
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_model)),
        'MAE': mean_absolute_error(y_test, y_pred_model),
        'R²': r2_score(y_test, y_pred_model)
    })

# Tampilkan tabel perbandingan performa model
results_df = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
display(results_df.round(4))

# Visualisasikan perbandingan RMSE antarmodel
plt.figure(figsize=(8, 5))
plt.bar(results_df['Model'], results_df['RMSE'], color=['teal', 'cornflowerblue', 'darkorange'])
plt.title('Perbandingan RMSE Model')
plt.xlabel('Model')
plt.ylabel('RMSE (lebih rendah lebih baik)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Visualisasikan perbandingan skor R² antarmodel
plt.figure(figsize=(8, 5))
plt.bar(results_df['Model'], results_df['R²'], color=['teal', 'cornflowerblue', 'darkorange'])
plt.title('Perbandingan Skor R² Model')
plt.xlabel('Model')
plt.ylabel('Skor R² (lebih tinggi lebih baik)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Laporkan model dengan RMSE terendah pada data uji
best_model = results_df.iloc[0]
print(f"Model dengan RMSE terendah: {best_model['Model']}")
print(f"RMSE: {best_model['RMSE']:.4f} | MAE: {best_model['MAE']:.4f} | R²: {best_model['R²']:.4f}")

## Kesimpulan dan Pekerjaan Mendatang

Dalam notebook ini, saya memulai analisis dari rasa ingin tahu untuk memahami dampak penggunaan kecerdasan buatan (AI) terhadap pelajar. Saya memilih dataset ini karena ingin mengetahui bagaimana penggunaan AI berkaitan dengan hasil akademik pelajar, khususnya perubahan IPK sebelum dan setelah semester. Melalui analisis data eksploratif, saya menelaah pola dan hubungan dalam data, kemudian membangun model regresi linear untuk memprediksi IPK setelah semester (*Post-Semester GPA*) berdasarkan fitur akademik dan penggunaan AI yang dipilih.

Kinerja model, yang dirangkum melalui nilai RMSE dan skor R², dapat menjadi titik awal untuk analisis lebih lanjut. Namun, hasil analisis ini menunjukkan hubungan dalam dataset dan tidak dengan sendirinya membuktikan bahwa penggunaan AI menyebabkan perubahan hasil akademik.

Pekerjaan mendatang dapat mencakup:
- Menjelajahi model yang lebih kompleks atau metode ansambel untuk meningkatkan akurasi prediksi.
- Menambahkan fitur lain, seperti keterampilan rekayasa prompt (*Prompt Engineering Skill*) atau keragaman alat (*Tool Diversity*), setelah variabel kategoris dikodekan.
- Menyelidiki kemungkinan interaksi antara kebiasaan belajar konvensional dan ketergantungan pada AI.

